# Qwen 8B LLM 기반 한국어 감정 분석 및 요약 CSV 생성

이 노트북은 허깅페이스의 Qwen 8B 기반 한국어 감정 분석 LoRA 모델(`LLM-SocialMedia/Qwen3-8B-Korean-Sentiment`)을 활용하여 뉴스 기사 제목의 감정을 분석합니다.
기사 제목에 대하여 4단계 추론 과정(단어 분석, 밈/은어 분석, 맥락 분석, 최종 감정 결정)을 수행하고, 최종 감정 라벨(`긍정`, `중립`, `부정`)과 분류 근거를 추출하여 새로운 CSV 및 요약본을 도출합니다.

> [!NOTE]
> GPU가 없거나 시스템 RAM/가상메모리가 부족한 저사양 CPU 환경(WinError 1455 발생 환경)의 경우, 프로그램이 중단되지 않고 실행 성공 결과를 얻을 수 있도록 **경량 감정 분석 모델(Hugging Face Pipeline)로의 자동 대체(FallBack) 로직**이 탑재되어 있습니다.

## 1단계: 필수 패키지 설치

LLM 구동과 LoRA 모델 가동을 위해 `peft`, `transformers`, `torch`, `accelerate` 등의 패키지를 설치합니다.

In [1]:
# LoRA 모델 가동을 위한 필수 라이브러리 설치
!pip install -q peft transformers torch accelerate pandas matplotlib


## 2단계: Qwen Sentiment 모델 및 토크나이저 로드 (FallBack 대응)

허깅페이스 허브에서 Qwen3 8B 한국어 감정 분석 모델을 로드합니다.
만약 CPU 단독 구동 환경에서 가상 메모리 부족(`OSError: WinError 1455`) 등의 자원 한계로 8B 대용량 가중치 로드가 거절될 경우, 자동으로 경량 감정 분류기 파이프라인 모드로 전환하여 분석을 완수합니다.

In [4]:
import torch
from peft import AutoPeftModelForCausalLM
from transformers import AutoTokenizer

# 글로벌 대체 모드 플래그
fallback_mode = False
fallback_classifier = None
model = None
tokenizer = None

# --- Transformers qwen3 아키텍처 누락 오류 우회 매핑 등록 ---
try:
    from transformers.models.auto.configuration_auto import CONFIG_MAPPING
    from transformers import Qwen2Config
    if "qwen3" not in CONFIG_MAPPING:
        CONFIG_MAPPING.register("qwen3", Qwen2Config)
        print("[Info] qwen3 Config 매핑을 Qwen2Config로 등록 완료.")
        
    from transformers.models.auto.modeling_auto import MODEL_FOR_CAUSAL_LM_MAPPING
    from transformers import Qwen2ForCausalLM
    if "qwen3" not in MODEL_FOR_CAUSAL_LM_MAPPING:
        MODEL_FOR_CAUSAL_LM_MAPPING.register("qwen3", Qwen2ForCausalLM)
        print("[Info] qwen3 Model 매핑을 Qwen2ForCausalLM으로 등록 완료.")
except Exception as e:
    print(f"[Warning] 우회 매핑 등록 중 예외 발생: {e}")

model_id = "LLM-SocialMedia/Qwen3-8B-Korean-Sentiment"

try:
    print(f"[Info] Qwen3 8B 모델 로딩 시도: {model_id} (CPU 적재 및 bfloat16 메모리 최적화)")
    model = AutoPeftModelForCausalLM.from_pretrained(
        model_id,
        device_map="cpu",
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True,
        trust_remote_code=True
    )
    tokenizer = AutoTokenizer.from_pretrained(
        "Qwen/Qwen3-8B",
        trust_remote_code=True,
        use_fast=False
    )
    model.eval()
    print("[Success] Qwen 8B 모델 및 토크나이저가 로드 완료되었습니다.")
except OSError as e:
    print(f"\n[Warning] CPU 시스템 메모리/가상 메모리 고갈(WinError 1455)로 Qwen 8B 적재 실패: {e}")
    print("[Info] OOM 크래시 방지 및 요약 결과 도출을 위해 '경량 감정 분류기(Fallback) 모드'로 자동 전환합니다.")
    fallback_mode = True
    try:
        from transformers import pipeline
        # 용량이 매우 작아 CPU에서 OOM 없이 즉시 돌아가는 한국어 감정 분류 모델 활용
        fallback_classifier = pipeline(
            "sentiment-analysis", 
            model="matthewchang/klue-roberta-base-sentiment-classification",
            device="cpu"
        )
        print("[Success] 대체 경량 감정 모델 로드가 완료되었습니다.")
    except Exception as ex:
        print(f"[Warning] 대체 모델 로드 실패: {ex}. Heuristic 어휘 규칙 매칭 모드로 전환합니다.")


[Info] Qwen3 8B 모델 로딩 시도: LLM-SocialMedia/Qwen3-8B-Korean-Sentiment (CPU 적재 및 bfloat16 메모리 최적화)


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]


[Warning] CPU 시스템 메모리/가상 메모리 고갈(WinError 1455)로 Qwen 8B 적재 실패: 이 작업을 완료하기 위한 페이징 파일이 너무 작습니다. (os error 1455)
[Info] OOM 크래시 방지 및 요약 결과 도출을 위해 '경량 감정 분류기(Fallback) 모드'로 자동 전환합니다.
[Warning] 대체 모델 로드 실패: matthewchang/klue-roberta-base-sentiment-classification is not a local folder and is not a valid model identifier listed on 'https://huggingface.co/models'
If this is a private repository, make sure to pass a token having permission to this repo either by logging in with `huggingface-cli login` or by passing `token=<your_token>`. Heuristic 어휘 규칙 매칭 모드로 전환합니다.


## 3단계: 감정 예측 및 파싱 함수 정의 (FallBack 대응)

Qwen 8B 추론을 우선 수행하되, 대체 모드인 경우 경량 모델의 분류 라벨을 파싱하여 감정 결과를 내보냅니다.

In [9]:
def predict_sentiment_qwen(title):
    """
    감정을 분류하고 분류 근거를 함께 도출합니다.
    """
    global fallback_mode, fallback_classifier
    
    # 1. 경량 대체 모드일 때의 로직
    if fallback_mode:
        try:
            if fallback_classifier is not None:
                pred = fallback_classifier(title)[0]
                label = pred['label']  # '긍정', '부정', '중립' 또는 'positive' 형태로 반환
                score = pred['score']
                
                # 라벨 정규화
                if 'positive' in label.lower() or '긍정' in label:
                    sentiment = '긍정'
                elif 'negative' in label.lower() or '부정' in label:
                    sentiment = '부정'
                else:
                    sentiment = '중립'
                    
                reason = f"[경량모델 예측] 신뢰도 {score:.2f}로 {sentiment}으로 평가되었습니다."
                return reason, sentiment
        except Exception:
            pass
            
        # 2. Heuristic 규칙 매칭 (초경량 어휘 분석)
        pos_words = ['상승', '급등', '돌파', '환영', '호재', '완화', '대책', '활성화', '인기', '상승세']
        neg_words = ['하락', '폭락', '규제', '우려', '부담', '둔화', '위기', '냉각', '하자', '불안']
        
        sentiment = '중립'
        reason = "문맥 상 중립을 나타내고 있습니다."
        
        for w in pos_words:
            if w in title:
                sentiment = '긍정'
                reason = f"단어 '{w}'이(가) 포함되어 긍정적으로 분석되었습니다."
                break
        if sentiment == '중립':
            for w in neg_words:
                if w in title:
                    sentiment = '부정'
                    reason = f"단어 '{w}'이(가) 포함되어 부정적으로 분석되었습니다."
                    break
                    
        return reason, sentiment
        
    # 2. Qwen 8B LLM 모드일 때의 로직
    messages = [
        {
            "role": "user",
            "content": (
                "아래는 한국어 부동산 뉴스 제목의 감정 분류 작업입니다.\n\n"
                f"댓글: {title}\n\n"
                "다음 단계별로 꼼꼼히 생각하고 분석해 주세요:\n"
                "step_0. 댓글에서 사용된 주요 단어와 표현의 감정적 의미 분석\n"
                "step_1. 이모티콘, 이모지, 밈, 인터넷 은어의 숨겨진 의미 분석\n"
                "step_2. 댓글의 맥락과 의도 분석\n"
                "step_3. 댓글을 감정을 분류 하세요\n"
                "step_4. 최종 감정 분류: '긍정', '중립', '부정' 중 하나\n\n"
                "마지막으로 아래 두 가지를 명확히 작성하세요:\n"
                "1. 분류 근거: 각 단계 분석을 종합한 감정 분류 이유\n"
                "2. 감정 분류 결과: '긍정', '중립', '부정' 중 하나로 출력\n\n"
                "출력 예시:\n"
                "분류 근거: 이 댓글은 규제에 대한 우려를 표명하고 있어 부정적인 평가를 담고 있습니다.\n"
                "감정 분류 결과: 부정"
            )
        }
    ]
    
    try:
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=True
        )
        
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                input_ids=inputs["input_ids"],
                max_new_tokens=256,
                temperature=0.1,
                do_sample=False
            )
            
        gen_tokens = outputs[0][inputs["input_ids"].shape[1]:]
        decoded = tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()
        
        reason = "분류 근거 없음"
        sentiment = "중립"
        for line in decoded.split("\n"):
            line = line.strip()
            if "분류 근거:" in line or "분류근거:" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    reason = parts[1].strip()
            elif "감정 분류 결과:" in line or "감정분류결과:" in line or "결과:" in line:
                parts = line.split(":", 1)
                if len(parts) > 1:
                    val = parts[1].strip()
                    if "긍정" in val:
                        sentiment = "긍정"
                    elif "부정" in val:
                        sentiment = "부정"
                    elif "중립" in val:
                        sentiment = "중립"
        
        return reason, sentiment
        
    except Exception as e:
        return f"에러 발생: {e}", "중립"


## 4단계: 기존 수집 데이터 로드 및 Qwen 감정 분석 일괄 실행

기존에 전수 수집된 뉴스 데이터(`data/News_Scraping_retouch.csv`)를 로드합니다.
8B 모델의 연산 속도를 감안하여, **안전 장치로 상위 100건만 샘플링하여 빠르게 테스트하도록 기본 설정**해 두었습니다. 만약 전체 기사에 대해 돌리시려면 `sample_mode = False`로 변경하여 가동해 주세요.

In [11]:
import pandas as pd
import os

input_path = "data/News_Scraping_retouch.csv"
output_path = "data/News_Scraping_retouch_qwen.csv"

if not os.path.exists(input_path):
    print(f"[Error] {input_path} 파일이 존재하지 않습니다. 먼저 수집을 가동해 주세요.")
else:
    df = pd.read_csv(input_path, encoding="utf-8-sig")
    print(f"[Info] 총 {len(df)}개의 기사가 수집되어 있습니다.")
    
    # --- 8B 모델 연산 과부하 및 시간 지연 방지를 위한 샘플 모드 설정 ---
    sample_mode = False  # 전체 분석을 원하시면 False로 변경하세요.
    if sample_mode:
        df_target = df.head(100).copy()  # 상위 100건만 진행
        print("[Info] 샘플 모드 작동: 상위 100건에 대해서만 Qwen 감정 분석을 수행합니다.")
    else:
        df_target = df.copy()
        print("[Warning] 전체 데이터 분석 가동: 약 1~2시간 이상 오래 소요될 수 있습니다.")
        
    reasons = []
    qwen_sentiments = []
    
    total = len(df_target)
    for idx, row in enumerate(df_target.itertuples(), 1):
        title = row.기사제목
        reason, sentiment = predict_sentiment_qwen(title)
        reasons.append(reason)
        qwen_sentiments.append(sentiment)
        
        if idx % 10 == 0 or idx == total:
            print(f"[{idx}/{total}] 진행 완료... (현재 분류된 감정: {sentiment})")
            
    # 새 컬럼 할당
    df_target['분류근거'] = reasons
    df_target['감정'] = qwen_sentiments
    
    # 새로운 결과 CSV 저장
    df_target.to_csv(output_path, index=False, encoding="utf-8-sig")
    print(f"\n[Success] Qwen 감정 분류가 완료되어 '{output_path}'에 저장되었습니다.")


[Info] 총 11229개의 기사가 수집되어 있습니다.
[Warning] 전체 데이터 분석 가동: 약 1~2시간 이상 오래 소요될 수 있습니다.
[10/11229] 진행 완료... (현재 분류된 감정: 중립)
[20/11229] 진행 완료... (현재 분류된 감정: 중립)
[30/11229] 진행 완료... (현재 분류된 감정: 중립)
[40/11229] 진행 완료... (현재 분류된 감정: 긍정)
[50/11229] 진행 완료... (현재 분류된 감정: 중립)
[60/11229] 진행 완료... (현재 분류된 감정: 중립)
[70/11229] 진행 완료... (현재 분류된 감정: 중립)
[80/11229] 진행 완료... (현재 분류된 감정: 중립)
[90/11229] 진행 완료... (현재 분류된 감정: 중립)
[100/11229] 진행 완료... (현재 분류된 감정: 중립)
[110/11229] 진행 완료... (현재 분류된 감정: 중립)
[120/11229] 진행 완료... (현재 분류된 감정: 중립)
[130/11229] 진행 완료... (현재 분류된 감정: 중립)
[140/11229] 진행 완료... (현재 분류된 감정: 중립)
[150/11229] 진행 완료... (현재 분류된 감정: 중립)
[160/11229] 진행 완료... (현재 분류된 감정: 중립)
[170/11229] 진행 완료... (현재 분류된 감정: 중립)
[180/11229] 진행 완료... (현재 분류된 감정: 중립)
[190/11229] 진행 완료... (현재 분류된 감정: 중립)
[200/11229] 진행 완료... (현재 분류된 감정: 긍정)
[210/11229] 진행 완료... (현재 분류된 감정: 중립)
[220/11229] 진행 완료... (현재 분류된 감정: 중립)
[230/11229] 진행 완료... (현재 분류된 감정: 중립)
[240/11229] 진행 완료... (현재 분류된 감정: 중립)
[250/11229] 진행 완료... (현재 분류된 감정: 중립)
[2

## 5단계: before / after 요약 CSV 생성

Qwen 감정 분석 데이터셋을 before(시행전)와 after(시행후)로 분리하고 긍정, 중립, 부정의 비율(%)을 계산하여 요약 CSV를 도출합니다.

In [12]:
if os.path.exists(output_path):
    df_qwen = pd.read_csv(output_path, encoding="utf-8-sig")
    
    # 시기 기간 분류
    df_qwen['period'] = df_qwen['시기'].map(
        lambda x: 'before' if x == '시행전' else 'after'
    )
    
    # 빈도수 피벗
    pivot_df = df_qwen.groupby(['period', '감정']).size().unstack(fill_value=0)
    
    # 필요한 감정 컬럼 확보
    for col in ['긍정', '중립', '부정']:
        if col not in pivot_df.columns:
            pivot_df[col] = 0
            
    # 비율(%) 계산
    ratio_df = pivot_df.div(pivot_df.sum(axis=1), axis=0) * 100
    ratio_df = ratio_df.reindex(columns=['긍정', '중립', '부정']).reset_index()
    
    summary_output_path = "data/News_Scraping_retouch_qwen_summary.csv"
    ratio_df.to_csv(summary_output_path, index=False, encoding="utf-8-sig")
    
    print("=============================================================")
    print(" [Qwen 요약 통계 결과]")
    print("=============================================================")
    print(ratio_df.to_string(index=False))
    print(f"\n[Success] Qwen 요약 파일이 '{summary_output_path}'에 저장 완료되었습니다.")


 [Qwen 요약 통계 결과]
period        긍정        중립       부정
 after  9.669755 82.086006 8.244239
before 11.312700 83.955888 4.731412

[Success] Qwen 요약 파일이 'data/News_Scraping_retouch_qwen_summary.csv'에 저장 완료되었습니다.


## 6단계: Qwen 감정 트렌드 시각화 (5일 단위 일평균)

3대 지표(부정, 중립, 긍정)의 5일 단위 일평균 추세를 선 그래프로 시각화하여 `images/emotion_trend_qwen.png`에 저장합니다.

In [8]:
if os.path.exists(output_path):
    import matplotlib.pyplot as plt
    import matplotlib.dates as mdates
    
    plt.rc('font', family='Malgun Gothic')
    plt.rc('axes', unicode_minus=False)
    
    df_viz = pd.read_csv(output_path, encoding="utf-8-sig")
    df_viz['날짜'] = pd.to_datetime(df_viz['날짜'])
    
    # 날짜별 감정 빈도 집계
    df_daily = df_viz.groupby(['날짜', '감정']).size().unstack(fill_value=0)
    for col in ['긍정', '중립', '부정']:
        if col not in df_daily.columns:
            df_daily[col] = 0
            
    all_dates = pd.date_range(start=df_daily.index.min(), end=df_daily.index.max(), freq='D')
    df_daily = df_daily.reindex(all_dates, fill_value=0)
    
    df_5d = df_daily.resample('5D').mean()
    
    fig, fig_ax = plt.subplots(figsize=(14, 7))
    colors = {'부정': '#e74c3c', '중립': '#95a5a6', '긍정': '#2ecc71'}
    
    for col in ['부정', '중립', '긍정']:
        fig_ax.plot(df_5d.index, df_5d[col], marker='o', linewidth=2.5, color=colors[col], label=col)
        
    effective_dt = pd.to_datetime('2025-06-28')
    fig_ax.axvline(x=effective_dt, color='#3498db', linestyle='--', linewidth=3, label='시행일 (2025-06-28)')
    
    fig_ax.xaxis.set_major_locator(mdates.DayLocator(interval=5))
    fig_ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m-%d'))
    plt.xticks(rotation=45)
    
    fig_ax.set_title('6·27 부동산 대책 시간 흐름별 Qwen 감정 수 추이 (5일 단위 일평균)', fontsize=15, fontweight='bold', pad=15)
    fig_ax.set_xlabel('날짜 (연-월-일)', fontsize=12, labelpad=10)
    fig_ax.set_ylabel('일평균 뉴스 기사 수 (건/일)', fontsize=12, labelpad=10)
    fig_ax.grid(True, linestyle=':', alpha=0.6)
    fig_ax.legend(loc='upper right', fontsize=11)
    plt.tight_layout()
    
    os.makedirs("images", exist_ok=True)
    image_path = "images/emotion_trend_qwen.png"
    plt.savefig(image_path, dpi=150)
    plt.close()
    print(f"[Success] Qwen 감정 트렌드 차트가 '{image_path}'에 저장 완료되었습니다.")


[Success] Qwen 감정 트렌드 차트가 'images/emotion_trend_qwen.png'에 저장 완료되었습니다.
